# SmartBite PP-OCRv5 Detection Fine-Tuning (Products-Real)

This notebook follows the same workflow style as your Date-Synth notebook, but fine-tunes **PP-OCRv5 detection**.

Workflow:
1. Mount Drive
2. Unzip `ppocrv5_product_real_dataset.zip`
3. Convert SmartBite `annotations.json` into PaddleOCR detection label format
4. Validate generated detection labels
5. Prepare PaddleOCR + offline wheelhouse dependencies
6. Run PP-OCRv5 detection training
7. Save training artifacts back to Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clean previous workspace
!rm -rf /content/dataset /content/workdir /content/output
!mkdir -p /content/dataset /content/workdir /content/output


In [ ]:
# Adjust if your zip is in another Drive location
DATASET_ZIP = "/content/drive/My Drive/sb-colab/ppocrv5_product_real_dataset.zip"

!unzip -q "${DATASET_ZIP}" -d /content/dataset/
!find /content/dataset -maxdepth 3 -type d | sed -n '1,120p'


In [ ]:
from pathlib import Path
import json

BASE_UNZIP_DIR = Path('/content/dataset')

# Robust auto-discovery: find any directory that has both
# train/annotations.json and evaluation/annotations.json
candidate_roots = []
for train_ann in BASE_UNZIP_DIR.rglob('train/annotations.json'):
    root = train_ann.parent.parent
    eval_ann = root / 'evaluation' / 'annotations.json'
    if eval_ann.exists():
        candidate_roots.append(root)

if not candidate_roots:
    # Helpful debug output before failing
    print('Could not auto-discover dataset root. Top-level dirs under /content/dataset:')
    for d in sorted(BASE_UNZIP_DIR.glob('*')):
        if d.is_dir():
            print(' -', d)
    raise FileNotFoundError(
        'Could not find dataset root containing train/annotations.json and evaluation/annotations.json. '
        'Check DATASET_ZIP path and unzip output.'
    )

PRODUCTS_REAL_ROOT = sorted(candidate_roots, key=lambda p: len(str(p)))[0]

TRAIN_ANN = PRODUCTS_REAL_ROOT / 'train' / 'annotations.json'
TRAIN_IMG_DIR = PRODUCTS_REAL_ROOT / 'train' / 'images'
EVAL_ANN = PRODUCTS_REAL_ROOT / 'evaluation' / 'annotations.json'
EVAL_IMG_DIR = PRODUCTS_REAL_ROOT / 'evaluation' / 'images'

assert TRAIN_ANN.exists(), f'Missing {TRAIN_ANN}'
assert EVAL_ANN.exists(), f'Missing {EVAL_ANN}'
assert TRAIN_IMG_DIR.exists(), f'Missing {TRAIN_IMG_DIR}'
assert EVAL_IMG_DIR.exists(), f'Missing {EVAL_IMG_DIR}'

print('PRODUCTS_REAL_ROOT =', PRODUCTS_REAL_ROOT)
print('TRAIN_ANN =', TRAIN_ANN)
print('EVAL_ANN  =', EVAL_ANN)


In [ ]:
from pathlib import Path
import json

WORKDIR = Path('/content/workdir/smartbite_ppocrv5_det_products_real')
WORKDIR.mkdir(parents=True, exist_ok=True)

train_label_file = WORKDIR / 'train_det_label.txt'
val_label_file = WORKDIR / 'val_det_label.txt'

# We keep all annotated classes as text regions for detector training.
# Paddle detection label format per line:
# image_path	[{"transcription":"...","points":[[x1,y1],[x2,y1],[x2,y2],[x1,y2]]}, ...]

def smartbite_to_det_labels(ann_path: Path, image_dir: Path, out_label_path: Path) -> tuple[int, int, int]:
    data = json.loads(ann_path.read_text(encoding='utf-8'))

    written = 0
    skipped_missing = 0
    skipped_no_ann = 0

    with out_label_path.open('w', encoding='utf-8') as f:
        for image_name in sorted(data.keys()):
            meta = data[image_name]
            image_path = image_dir / image_name

            if not image_path.exists():
                skipped_missing += 1
                continue

            anns = meta.get('ann', [])
            records = []
            for ann in anns:
                bbox = ann.get('bbox')
                if not isinstance(bbox, list) or len(bbox) != 4:
                    continue
                x1, y1, x2, y2 = bbox
                if x2 <= x1 or y2 <= y1:
                    continue

                transcription = ann.get('transcription')
                if not isinstance(transcription, str) or not transcription.strip():
                    transcription = ann.get('cls', 'text')

                records.append({
                    'transcription': transcription,
                    'points': [
                        [float(x1), float(y1)],
                        [float(x2), float(y1)],
                        [float(x2), float(y2)],
                        [float(x1), float(y2)],
                    ],
                })

            if not records:
                skipped_no_ann += 1
                continue

            f.write(f"{image_path}	{json.dumps(records, ensure_ascii=False)}\n")
            written += 1

    return written, skipped_missing, skipped_no_ann

train_written, train_missing, train_noann = smartbite_to_det_labels(TRAIN_ANN, TRAIN_IMG_DIR, train_label_file)
val_written, val_missing, val_noann = smartbite_to_det_labels(EVAL_ANN, EVAL_IMG_DIR, val_label_file)

print('train_det_label:', train_label_file)
print('val_det_label  :', val_label_file)
print('train_written=', train_written, 'missing=', train_missing, 'no_ann=', train_noann)
print('val_written  =', val_written, 'missing=', val_missing, 'no_ann=', val_noann)


In [ ]:
from pathlib import Path

print('WORKDIR:', WORKDIR)
for split, p in [('train', train_label_file), ('val', val_label_file)]:
    lines = [line for line in p.read_text(encoding='utf-8').splitlines() if line.strip()]
    print(f'{split}: label_lines={len(lines)}')
    if lines:
        print(f'{split} sample:', lines[0][:260], '...')


## Wheelhouse (No Internet Installs in Colab)

This follows your existing Date-Synth approach:
- use PaddleOCR zip from Drive (no git clone)
- install Paddle + requirements from pre-downloaded wheelhouse zip


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

# Use PaddleOCR archive from Drive.
PADDELOCR_ZIP = Path('/content/drive/My Drive/sb-colab/PaddleOCR.zip')
PADDELOCR_DIR = Path('/content/PaddleOCR')
if not PADDELOCR_DIR.exists():
    assert PADDELOCR_ZIP.exists(), f'Missing PaddleOCR zip: {PADDELOCR_ZIP}'
    !unzip -q "{PADDELOCR_ZIP}" -d /content

%cd /content/PaddleOCR

# Avoid slow model-source checks.
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'

has_gpu = os.system('nvidia-smi > /dev/null 2>&1') == 0
print('GPU available:', has_gpu)

wheel_zip = Path('/content/drive/My Drive/colab-cu118.zip' if has_gpu else '/content/drive/My Drive/colab-cpu.zip')
wheelhouse = Path('/content/wheels/colab-cu118' if has_gpu else '/content/wheels/colab-cpu')
if not wheelhouse.exists():
    assert wheel_zip.exists(), f'Missing wheelhouse zip: {wheel_zip}'
    wheelhouse.parent.mkdir(parents=True, exist_ok=True)
    !unzip -q "{wheel_zip}" -d /content/wheels

assert wheelhouse.exists(), f'Wheelhouse did not extract correctly: {wheelhouse}'
print('Using wheelhouse:', wheelhouse)

def run_live(cmd, env=None):
    print('>>', shlex.join(cmd))
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

def pip_install(args):
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(wheelhouse), *args]
    run_live(cmd)

if has_gpu:
    pip_install(['paddlepaddle-gpu'])
else:
    pip_install(['paddlepaddle'])

pip_install(['-r', '/content/PaddleOCR/requirements.txt'])

import paddle
print('paddle version:', paddle.__version__)
print('compiled_with_cuda:', paddle.is_compiled_with_cuda())


In [ ]:
from pathlib import Path

# Detection base config
BASE_CONFIG = Path('/content/PaddleOCR/configs/det/PP-OCRv5/PP-OCRv5_server_det.yml')
assert BASE_CONFIG.exists(), f'Missing config: {BASE_CONFIG}'

# Pretrained det weights: adjust if needed.
# We try common names first.
candidates = [
    Path('/content/drive/My Drive/sb-colab/models/PP-OCRv5_server_det_pretrained.pdparams'),
    Path('/content/drive/My Drive/models/PP-OCRv5_server_det_pretrained.pdparams'),
]
PRETRAINED_MODEL = None
for c in candidates:
    if c.exists():
        PRETRAINED_MODEL = c
        break

assert PRETRAINED_MODEL is not None, (
    'Could not find PP-OCRv5 detection pretrained model in Drive. '
    'Put it under /content/drive/My Drive/sb-colab/models/PP-OCRv5_server_det_pretrained.pdparams '
    'or update PRETRAINED_MODEL path in this cell.'
)

OUTPUT_DIR = Path('/content/output/ppocrv5_product_real_det_run')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 20
BATCH_SIZE = 16
LEARNING_RATE = 0.001

print('BASE_CONFIG     =', BASE_CONFIG)
print('PRETRAINED_MODEL=', PRETRAINED_MODEL)
print('OUTPUT_DIR      =', OUTPUT_DIR)
print('TRAIN_LABEL     =', train_label_file)
print('VAL_LABEL       =', val_label_file)


In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

def run_live(cmd, env=None):
    print(shlex.join(cmd))
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

cmd = [
    sys.executable,
    '/content/PaddleOCR/tools/train.py',
    '-c', str(BASE_CONFIG),
    '-o',
    f'Global.pretrained_model={PRETRAINED_MODEL}',
    f'Global.save_model_dir={OUTPUT_DIR}',
    f'Global.epoch_num={EPOCHS}',
    'Global.print_batch_step=10',
    'Global.eval_batch_step=[0,500]',
    'Global.save_epoch_step=1',
    f'Optimizer.lr.learning_rate={LEARNING_RATE}',
    f'Train.loader.batch_size_per_card={BATCH_SIZE}',
    f'Eval.loader.batch_size_per_card=1',
    f'Train.dataset.label_file_list=["{train_label_file}"]',
    f'Eval.dataset.label_file_list=["{val_label_file}"]',
]

if os.system('nvidia-smi > /dev/null 2>&1') != 0:
    cmd.append('Global.use_gpu=False')

latest_base = Path(OUTPUT_DIR) / 'latest'
latest_params = Path(str(latest_base) + '.pdparams')
if latest_params.exists():
    cmd.append(f'Global.checkpoints={latest_base}')
    print('Resuming from:', latest_base)

env = dict(os.environ)
env['PYTHONPATH'] = '/content/PaddleOCR'
env['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'

run_live(cmd, env=env)

print('Detection training finished.')
print('Artifacts in:', OUTPUT_DIR)


In [ ]:
from pathlib import Path

print('OUTPUT_DIR:', OUTPUT_DIR)
print('exists:', OUTPUT_DIR.exists())
print('files:', [p.name for p in sorted(OUTPUT_DIR.glob('*'))[:30]])

train_log = OUTPUT_DIR / 'train.log'
print('train.log exists:', train_log.exists())
if train_log.exists():
    lines = train_log.read_text(errors='ignore').splitlines()
    for line in lines[-15:]:
        print(line)


In [ ]:
from pathlib import Path
import shutil

# Keep this path as you prefer. Update if needed.
FINAL_MODEL_DRIVE_DIR = Path('/content/drive/My Drive/sb-colab/ppocrv5_product_real_det_run')

assert OUTPUT_DIR.exists(), f'Missing OUTPUT_DIR: {OUTPUT_DIR}'
assert FINAL_MODEL_DRIVE_DIR.parent.exists(), f'Missing parent dir: {FINAL_MODEL_DRIVE_DIR.parent}'

FINAL_MODEL_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

for item in OUTPUT_DIR.iterdir():
    target = FINAL_MODEL_DRIVE_DIR / item.name
    if item.is_dir():
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)

print('Saved detection model artifacts to:', FINAL_MODEL_DRIVE_DIR)


## Notes
- This notebook fine-tunes **detection only**.
- You can keep recognition fine-tuning as a separate notebook/process.
- If you want to ignore some classes (e.g., use only date-like boxes), filter them in the conversion cell before writing labels.


In [ ]:
# Optional: continue fine-tuning from current best with lower LR
# Run this only after the first full training completes.

EXTRA_EPOCHS = 10
LR_SCALE = 0.25  # use lower LR for continuation
CONTINUE_LR = float(LEARNING_RATE) * LR_SCALE
CONTINUE_OUTPUT_DIR = Path('/content/output/ppocrv5_product_real_det_run_extra')
BEST_BASE = OUTPUT_DIR / 'best_accuracy'
BEST_PARAMS = Path(str(BEST_BASE) + '.pdparams')

assert BEST_PARAMS.exists(), f'best checkpoint not found: {BEST_PARAMS}'
CONTINUE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    '/content/PaddleOCR/tools/train.py',
    '-c', str(BASE_CONFIG),
    '-o',
    f'Global.pretrained_model={BEST_BASE}',
    f'Global.save_model_dir={CONTINUE_OUTPUT_DIR}',
    f'Global.epoch_num={EXTRA_EPOCHS}',
    'Global.print_batch_step=10',
    'Global.eval_batch_step=[0,500]',
    'Global.save_epoch_step=1',
    f'Optimizer.lr.learning_rate={CONTINUE_LR}',
    f'Train.loader.batch_size_per_card={BATCH_SIZE}',
    'Eval.loader.batch_size_per_card=1',
    f'Train.dataset.label_file_list=["{train_label_file}"]',
    f'Eval.dataset.label_file_list=["{val_label_file}"]',
]

if os.system('nvidia-smi > /dev/null 2>&1') != 0:
    cmd.append('Global.use_gpu=False')

env = dict(os.environ)
env['PYTHONPATH'] = '/content/PaddleOCR'
env['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'

run_live(cmd, env=env)

print('Continuation training finished.')
print('Artifacts in:', CONTINUE_OUTPUT_DIR)
